### Extractor 

In [ ]:
import ffmpeg 
def extract_audio(video_path:str , audio_path:str):
    (
        ffmpeg.input(video_path).output(audio_path,ac=1,ar=16000)
        .overwrite_output()
        .run()
    )

### test 

In [5]:
extract_audio(
    video_path="sample.mp4",
    audio_path="audio.wav"
)
print('sucessfully')

sucessfully


ffmpeg version 9.0.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.3.34.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/9.0.2 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      61.  1.102 / 61.  1.102
  libavcodec     63.  1.102 / 63.  1.102
  libavformat    63.  1.102 / 63.  1.102
  libavdevice    63.  1.102 / 63.  1.102
  libavfilter    12.  1.102 / 12.  1.102
  libswscale     10.  1.102 / 10.  1.102
  libswresample   7.  1.102 /  7.  1.102
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'sample.mp4':
  Metadata:
    major_brand     : mp42
    minor_version   : 0
    compatible_brands: mp42isom
  Duration: 00:00:10.17, start: 0.0000

### wishper adding 

In [6]:
from faster_whisper  import WhisperModel 
model = WhisperModel("base",compute_type="int8")

def transcrib_audio(audip_path:str):
    segments , info = model.transcribe(audip_path)
    transcrib = ""
    
    for segment in segments :
        transcrib+=segment.text+''
    return transcrib.strip()

In [7]:
## Testing 
text = transcrib_audio("audio.wav")
print(text)

You probably don't find out about it anyway, so here's a little preemptive truth telling. There's no happy ending.


### Chucking add 

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
def split_into_chunks(transcrib:str):
    spliter = RecursiveCharacterTextSplitter(chunk_size=4000 , chunk_overlap=200)
    chunks = spliter.split_text(transcrib)
    return chunks 

## testing 


from chunking import split_into_chunks 
with open (file='transcrib.txt',mode='r') as f:
    content = f.read()

chunks = split_into_chunks(content)
print(f'the number of chunks : {len(chunks)}')
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk[:200])

the number of chunks : 1

--- Chunk 1 ---
You probably don't find out about it anyway, so here's a little preemptive truth telling. There's no happy ending.


## LLM Transfer 

In [12]:
from langchain_groq import ChatGroq 
from dotenv import load_dotenv 
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

def summarize(chunks:str):
    prompt = f"""
Summarize the following transcript into clear study notes.

Transcript:
{chunk}
"""

    response = llm.invoke(prompt)
    return response.content

# test 


with open('transcrib.txt','r') as f :
    transcrib = f.read 
    
summary_note = summarize(transcrib)
print(summary_note)

**Study Notes – Key Takeaways**

- **Preemptive Disclosure**  
  - The speaker chooses to reveal information before the listener discovers it on their own.  
  - This approach is intended to control the narrative and possibly mitigate surprise or misinformation.

- **Outcome Warning**  
  - The central message: *“There’s no happy ending.”*  
  - Implies that whatever situation or story is being referenced will not conclude positively.

- **Implications for the Audience**  
  - Expectation management: listeners should prepare for an unfavorable or unresolved outcome.  
  - Encourages critical thinking about the context and potential consequences before further involvement.

- **Possible Applications**  
  - Useful in scenarios where early warning can influence decisions (e.g., project risks, personal relationships, narrative analysis).  
  - Highlights the importance of transparent communication, even when the news is negative.
